## Test if measuring morality could ccorrelate with anthropomorphism
- pay attention to which parts of the sentence are used during the classification process
- do so with XAI methods of choice

In [1]:
import json
from pprint import pprint
import pandas as pd
from tqdm.auto import tqdm
from transformers import AutoTokenizer, AutoModelForCausalLM

import torch
device = torch.accelerator.current_accelerator().type if torch.accelerator.is_available() else "cpu"
print(f"Using {device} device")
from torch.quantization import quantize_dynamic
from torchao.quantization.quant_api import Int8DynActInt4WeightQuantizer


import spacy
from spacy.matcher import DependencyMatcher
#from spacy import displacy

nlp = spacy.load('de_core_news_lg')

Using mps device


W0503 02:00:53.340000 36431 site-packages/torch/distributed/elastic/multiprocessing/redirects.py:29] NOTE: Redirects are currently not supported in Windows or MacOs.


In [2]:
# load docs (KI sentences) to analyse
with open("generated_data/AI_sentences/Leipzig_KI_sentences_all.json", "r") as f:
    leipzig_data = json.load(f)
taz24_data = pd.read_csv("generated_data/AI_sentences/KI_sentences_translated.csv")
print("leipzig:", len(leipzig_data),list(leipzig_data.keys()), "\ntaz24:", taz24_data.columns)

leipzig: 31 ['2003_1M', '2013_1M', '2007_1M', '1996_1M', '2017_1M', '2023_1M', '2006_1M', '1997_1M', '2016_1M', '2022_1M', '2002_1M', '2012_1M', '2009_1M', '2019_1M', '1998_1M', '2008_1M', '2018_1M', '1999_1M', '2005_1M', '2015_1M', '2021_1M', '2001_1M', '2025_1M', '2011_1M', '2000_1M', '2010_1M', '2004_1M', '1995_1M', '2014_1M', '2020_1M', '2024'] 
taz24: Index(['id', 'original', 'translation'], dtype='str')


In [3]:
# load docs (anthropomorphic KI sentences) to analyse
with open("generated_data/Anthrop_sentences/all_anthro_ki.json", "r") as f:
    anthro_data = json.load(f)

print("sum", len(anthro_data['sb']) + len(anthro_data['ob']) + len(anthro_data['v']))
print("sum", len(anthro_data['sb']) , len(anthro_data['ob']) , len(anthro_data['v']))

sum 1008
sum 805 203 0


In [4]:
# measure using library?


In [5]:
# ask llm to generate RoT for each doc
# then analzse sb relationships for attention (to find actor (agent))

In [6]:
def get_sent_dep(dep_dict):
    sent = dep_dict['sentence']
    dep = dep_dict['dependent']
    return sent, dep

def get_sent_dep_txt(sentence):
    print("sentence:", sentence)
    doc = nlp(sentence)
    relation = {}
    if doc.has_annotation("DEP"):
        for token in doc:
            print("token:", token.text, "pos:", token.pos_, "dep:", token.dep_)
            if token.pos_ == "VERB":
                for child in token.children:
                    if child.dep_ in ["sb"]:
                        relation = {
                            "type": "verb_subject",
                            "verb": token.text,
                            "dependent": child.text,
                            "dep_label": child.dep_,
                            "verb_pos": token.pos_,
                            "dep_pos": child.pos_,
                            "sentence": doc.text
                        }
    sent, dep = get_sent_dep(relation)
    return relation, sent, dep

### LLAMA

In [7]:
def flatten(xss):
    return [x for xs in xss for x in xs]

leipzig_flat = flatten([list(leipzig_data[k].values()) for k in leipzig_data.keys()])
print("leipzig_flat:", len(leipzig_flat))

leipzig_flat: 16423


In [10]:
# model
tokenizer = AutoTokenizer.from_pretrained("meta-llama/Llama-3.3-70B-Instruct")
# model
tokenizer = AutoTokenizer.from_pretrained("meta-llama/Llama-3.3-70B-Instruct")
model = AutoModelForCausalLM.from_pretrained(
    "meta-llama/Llama-3.3-70B-Instruct",
    device_map="auto",
    load_in_8bit=True,
    llm_int8_enable_fp32_cpu_offload=True,
    trust_remote_code=True,
)
quantized_model = quantize_dynamic(
    model,
    {torch.nn.Linear},  # Specify which layers to quantize
    dtype=torch.qint8
)

The `load_in_4bit` and `load_in_8bit` arguments are deprecated and will be removed in the future versions. Please, pass a `BitsAndBytesConfig` object in `quantization_config` argument instead.


Loading checkpoint shards:   0%|          | 0/30 [00:00<?, ?it/s]

Some parameters are on the meta device because they were offloaded to the disk.
/var/folders/0h/lr036jk55nq7b5pmf0kg0qq40000gn/T/ipykernel_36431/3568006925.py:12: DeprecationWarning: torch.ao.quantization is deprecated and will be removed in 2.10. 
For migrations of users: 
1. Eager mode quantization (torch.ao.quantization.quantize, torch.ao.quantization.quantize_dynamic), please migrate to use torchao eager mode quantize_ API instead 
2. FX graph mode quantization (torch.ao.quantization.quantize_fx.prepare_fx,torch.ao.quantization.quantize_fx.convert_fx, please migrate to use torchao pt2e quantization API instead (prepare_pt2e, convert_pt2e) 
3. pt2e quantization has been migrated to torchao (https://github.com/pytorch/ao/tree/main/torchao/quantization/pt2e) 
see https://github.com/pytorch/ao/issues/2259 for more details
  quantized_model = quantize_dynamic(


RuntimeError: Tensor.item() cannot be called on meta tensors

In [ ]:
for sentence in tqdm(leipzig_flat, desc="generating RoT"): #leipzig_data["2025_1M"].items()
	text = sentence.strip()
	messages = [
		{"role": "user", "content":f"Definiere Faustregeln in der Form 'Es ist gut/schlecht, wenn ___ , weil ___.' Text:{text}"},
	]
	inputs = tokenizer.apply_chat_template(
		messages,
		add_generation_prompt=True,
		tokenize=True,
		return_dict=True,
		return_tensors="pt",
	).to(model.device)

	outputs = model.generate(**inputs, max_new_tokens=40)
	print("------------------------------------------------------------------------------")
	print(tokenizer.decode(outputs[0][inputs["input_ids"].shape[-1]:]))

### BERT

In [ ]:
tokenizer = AutoTokenizer.from_pretrained("google-bert/bert-base-german-cased")
model = AutoModelForCausalLM.from_pretrained("google-bert/bert-base-german-cased", is_decoder=True)

# prompt bert to gnerate RoT for each sentence in leipzig data
model.eval()
rot_sentences = {}

for id, sentence in tqdm(leipzig_flat, desc="generating RoT"):
    text = sentence.strip()
    #prompt = f"{text} Weil {tokenizer.mask_token} {tokenizer.mask_token} {tokenizer.mask_token}."
    prompt = f"""Definiere Faustregeln in der Form 'Es ist gut/schlecht, wenn ___ , weil ___.'
    Text:{text}
    Faustregel: """
    inputs = tokenizer(prompt, return_tensors="pt").to(model.device)
    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=200,
            temperature=0.7,
            do_sample=True,
            top_p=0.9,
        )
    rot_sentences[id] = tokenizer.decode(outputs[0], skip_special_tokens=True)


In [ ]:
pprint(rot_sentences)

In [ ]:
# then use get_sent_dep_txt to filter relations in the generated RoT as ifi they were single sentences
rot_dep = {}
for id,txt in rot_sentences.items():
    print("analyzing sentence:", txt)
    try:
        relation, sent, dep = get_sent_dep_txt(txt)
        rot_dep[id] = {"rel": relation, "sent": sent, "dep": dep}
    except:
        print("error analyzing sentence:", txt)
print(rot_dep)